# Lesson 1.5 — From linear regression to neural networks (and what BERT actually is)

Companion to [Lesson 1](lesson_01_linear_regression.ipynb).

After Lesson 1 you've trained a model with **2 parameters** that learns to fit a line. After Lesson 4 you train a tiny BERT with **3000 parameters** that does fill-in-the-blank. This notebook bridges the gap.

We'll answer two questions that should be on your mind:

1. **"Wait — is BERT just linear regression with more knobs? Or is it a different kind of model?"**
2. **"When do you need a neural network instead of linear regression?"**

The story we'll build today, on a single dataset:

| Approach | Parameters | Can it fit the curve? |
|---|---|---|
| **Linear regression** (L1 — `y = w*x + b`) | 2 | ❌ No |
| **Polynomial regression** (same loop, more features) | 3 | ✅ Yes (with handcrafted x² feature) |
| **Neural network** (Linear → ReLU → Linear) | ~30 | ✅ Yes (no feature engineering needed) |
| **Transformer** (BERT, PRAGMA) | thousands–billions | ✅ Yes, on sequences |

> 🔑 **The big takeaway:** the **5-line training loop is the same for all four**. What changes is the **model in the middle**. BERT and PRAGMA are NEURAL NETWORKS — specifically a kind called Transformers — NOT linear regression. The training recipe is shared; the model architecture is different.


## Step 0 — Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False)

## Step 1 — A new dataset: this one is a CURVE

In Lesson 1 the data was a straight line: `y = 2x + 1`. Linear regression nailed it because a line is exactly what linear regression can fit.

This time the secret rule is **a parabola**: `y = x² - 4x + 3`. Let's see if linear regression can still handle it.

In [ ]:
# Generate the data
x_data = torch.linspace(-2, 6, 40)
y_true = x_data ** 2 - 4 * x_data + 3 + torch.randn(40) * 0.5   # parabola + noise

print(f"x range: {x_data.min().item():.1f} to {x_data.max().item():.1f}")
print(f"y range: {y_true.min().item():.1f} to {y_true.max().item():.1f}")
print()

# Show first/last few points
print(f"{'x':>6}  {'y_true':>7}")
print("-" * 16)
for i in [0, 5, 10, 15, 20, 25, 30, 35, 39]:
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}")

**Look at the y values.** They go DOWN and then UP. That's not a line. No straight line could possibly fit all those points well — a line either goes up the whole way or down the whole way.

Let's draw it (ASCII):

In [ ]:
def ascii_scatter(x, y, marker='*', width=60, height=20, title=""):
    if title:
        print(title)
    xmin, xmax = x.min().item(), x.max().item()
    ymin, ymax = y.min().item(), y.max().item()
    grid = [[" "] * width for _ in range(height)]
    for xi, yi in zip(x.tolist(), y.tolist()):
        col = int((xi - xmin) / (xmax - xmin) * (width - 1))
        row = height - 1 - int((yi - ymin) / (ymax - ymin) * (height - 1))
        if 0 <= col < width and 0 <= row < height:
            grid[row][col] = marker
    print(f"{ymax:6.1f} |" + "".join(grid[0]))
    for row in grid[1:-1]:
        print("       |" + "".join(row))
    print(f"{ymin:6.1f} |" + "".join(grid[-1]))
    print("       +" + "-" * width)
    print(f"        x={xmin:.1f}{' ' * (width-12)}x={xmax:.1f}")

ascii_scatter(x_data, y_true, title="The data (a parabola):")

See the U-shape? That's our challenge. Linear regression can only draw STRAIGHT lines, and there is no straight line that hits all those points.

## Step 2 — Try linear regression (will fail)

Same model as Lesson 1: `y = w*x + b`. Two parameters. Let's see how badly it fails.

In [ ]:
# Model: y = w*x + b
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
opt = torch.optim.SGD([w, b], lr=0.001)

# Train
for _ in range(1000):
    y_pred = w * x_data + b
    loss = ((y_pred - y_true) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

print(f"Linear regression result: y = {w.item():.3f} * x + {b.item():.3f}")
print(f"Final loss: {loss.item():.3f}")
print()
print("(Compare to L1 which got loss ≈ 0.0001 on its line.)")
print()

# Predictions
y_pred_linear = (w * x_data + b).detach()
print("First few predictions vs truth:")
print(f"{'x':>6}  {'y_true':>7}  {'y_pred':>7}  {'error':>7}")
print("-" * 33)
for i in [0, 10, 20, 30, 39]:
    err = y_pred_linear[i].item() - y_true[i].item()
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}  {y_pred_linear[i].item():>7.2f}  {err:>+7.2f}")

**Look at the errors.** At the endpoints (low and high x) the model's predictions are way off. The best straight line just CAN'T fit a curve.

Loss settles around ~3-5, much worse than L1's ~0.0001 on its straight-line data.

## Step 3 — Trick: give linear regression more features

Here's the key insight that confuses everyone: **"linear regression" doesn't mean "fits a line in x"**. It means **"linear in the parameters"**. We can give it any transformation of x as a feature!

If we add `x²` as a feature, the model becomes:

$$y = w_1 \cdot x + w_2 \cdot x^2 + b$$

This is still LINEAR REGRESSION (3 parameters: w₁, w₂, b). But now it can fit a parabola.

This is called **polynomial regression**, but it's mathematically just linear regression with extra features.

In [ ]:
# Same SGD, more parameters
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b  = torch.tensor(0.0, requires_grad=True)
opt = torch.optim.SGD([w1, w2, b], lr=0.0005)

# Train
for _ in range(2000):
    y_pred = w1 * x_data + w2 * x_data**2 + b
    loss = ((y_pred - y_true) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

print(f"Polynomial regression result:")
print(f"  y = {w1.item():.3f} * x + {w2.item():.3f} * x² + {b.item():.3f}")
print(f"  (true rule was:  y =  -4.000 * x +  1.000 * x² + 3.000)")
print()
print(f"Final loss: {loss.item():.3f}    (was ~25 for the pure linear model — a 25x improvement)")
print()
y_pred_poly = (w1 * x_data + w2 * x_data**2 + b).detach()
print("First few predictions vs truth:")
print(f"{'x':>6}  {'y_true':>7}  {'y_pred':>7}  {'error':>7}")
print("-" * 33)
for i in [0, 10, 20, 30, 39]:
    err = y_pred_poly[i].item() - y_true[i].item()
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}  {y_pred_poly[i].item():>7.2f}  {err:>+7.2f}")

**Beautiful.** Loss drops dramatically. The model recovered the true coefficients (almost) perfectly:

- True: `y = -4x + x² + 3`
- Found: `y ≈ -4x + 1x² + 3`

**Same training loop. More features. Better model.** This is still linear regression — we just gave it more knobs and more inputs.

🤔 **But wait — there's a catch.** How would you know to add `x²`? What if the relationship was `sin(x)` or something more complex?

You'd have to TRY many features (x, x², x³, sin(x), e^x...) and see which ones help. For simple data this is doable. For complex data (text, images, banking events) — there are too many possible features to enumerate.

**That's where neural networks come in.**

## Step 4 — Enter the neural network

A neural network is a **stack of linear transformations with nonlinear activations in between**.

The simplest one (a **Multilayer Perceptron**, or MLP) looks like this:

```
input x
   │
   ▼
Linear  ──► 8 hidden numbers
   │
   ▼
ReLU    ──► same 8 numbers, but negative values clipped to 0
   │
   ▼
Linear  ──► 1 output number (the prediction)
```

The key insight: **the ReLU activation introduces NONLINEARITY**. Without it, two stacked linear layers would still produce a linear function (because composing linear functions gives a linear function). With ReLU in between, the network can model curves, kinks, and arbitrarily complex patterns.

**Universal approximation theorem (don't worry about the math):** with enough hidden units, an MLP can approximate ANY smooth function. You don't have to handcraft features like `x²` — the network learns them automatically.

In [ ]:
class MLP(nn.Module):
    """A tiny neural network: 1 input → 8 hidden units → 1 output."""
    def __init__(self, hidden=8):
        super().__init__()
        self.layer1 = nn.Linear(1, hidden)     # input → hidden
        self.layer2 = nn.Linear(hidden, 1)     # hidden → output

    def forward(self, x):
        h = self.layer1(x)
        h = F.relu(h)                          # the nonlinearity
        return self.layer2(h)

net = MLP(hidden=8)
total = sum(p.numel() for p in net.parameters())
print(f"Tiny MLP architecture:")
for name, p in net.named_parameters():
    shape_str = str(tuple(p.shape))
    print(f"  {name:<20s}  shape {shape_str:<10s}  {p.numel()} params")
print(f"\nTotal parameters: {total}")
print(f"(Compare to: 2 for linear regression, 3 for polynomial.)")

## Step 5 — Train the MLP on the SAME parabola data

Same training loop. Same loss function. Same optimizer. Only the model changed.

In [ ]:
net = MLP(hidden=8)
opt = torch.optim.Adam(net.parameters(), lr=0.05)

# We need to reshape x_data to (N, 1) — neural nets expect a batch dimension
x_in = x_data.unsqueeze(-1)     # (40, 1)
y_target = y_true.unsqueeze(-1) # (40, 1)

history = []
for step in range(2001):
    y_pred = net(x_in)
    loss = F.mse_loss(y_pred, y_target)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0:
        history.append((step, loss.item()))

print(f"{'step':>5}  {'loss':>8}")
print("-" * 16)
for step, l in history:
    print(f"{step:>5}  {l:>8.4f}")

# Get final predictions
y_pred_nn = net(x_in).squeeze(-1).detach()

**Loss converges similarly to the polynomial regression** — both end up around the noise floor (~0.25 due to the random noise we added).

The MLP figured out the parabolic shape **without being told to add x² as a feature**. It learned the right features on its own.

## Step 6 — Side-by-side comparison

All three models on the same data:

In [ ]:
print(f"{'x':>6}  {'y_true':>7}  {'linear':>7}  {'poly':>7}  {'NN':>7}")
print("-" * 40)
for i in [0, 5, 10, 15, 20, 25, 30, 35, 39]:
    print(f"{x_data[i].item():>6.2f}  {y_true[i].item():>7.2f}  "
          f"{y_pred_linear[i].item():>7.2f}  "
          f"{y_pred_poly[i].item():>7.2f}  "
          f"{y_pred_nn[i].item():>7.2f}")
print()
loss_linear = ((y_pred_linear - y_true) ** 2).mean().item()
loss_poly   = ((y_pred_poly   - y_true) ** 2).mean().item()
loss_nn     = ((y_pred_nn     - y_true) ** 2).mean().item()
print(f"Final MSE: linear={loss_linear:.3f}   polynomial={loss_poly:.3f}   neural net={loss_nn:.3f}")

**The linear model fails. The polynomial and the neural network both succeed.**

But notice: the polynomial uses **3 parameters**, the neural net uses **25**. The neural net is using extra capacity that it didn't strictly need for this simple problem.

So why ever use a neural network? Because for **complex data** (text, images, banking event sequences), you don't know what the right features are. You can't say "add x²" because the data is too complex for that intuition to apply. A neural network with enough capacity will figure out the features for you.

## Step 7 — Visualise all three models' fits

ASCII overlay so you can see them on the data:

In [ ]:
def ascii_overlay(x, y_true, predictions, markers, width=70, height=18):
    xmin, xmax = x.min().item(), x.max().item()
    all_y = torch.cat([y_true] + [p for p in predictions])
    ymin, ymax = all_y.min().item(), all_y.max().item()

    grid = [[' '] * width for _ in range(height)]
    # Plot truth as '.'
    for xi, yi in zip(x.tolist(), y_true.tolist()):
        col = int((xi - xmin) / (xmax - xmin) * (width - 1))
        row = height - 1 - int((yi - ymin) / (ymax - ymin) * (height - 1))
        if 0 <= col < width and 0 <= row < height:
            grid[row][col] = '·'
    # Plot each prediction
    for pred, marker in zip(predictions, markers):
        for xi, yi in zip(x.tolist(), pred.tolist()):
            col = int((xi - xmin) / (xmax - xmin) * (width - 1))
            row = height - 1 - int((yi - ymin) / (ymax - ymin) * (height - 1))
            if 0 <= col < width and 0 <= row < height and grid[row][col] == ' ':
                grid[row][col] = marker

    print(f"{ymax:6.1f} |" + "".join(grid[0]))
    for row in grid[1:-1]:
        print("       |" + "".join(row))
    print(f"{ymin:6.1f} |" + "".join(grid[-1]))
    print("       +" + "-" * width)
    print(f"       Legend:  ·  = true data  L = linear  P = polynomial  N = neural net")

print("All three models' predictions overlaid on the data:")
ascii_overlay(x_data, y_true, [y_pred_linear, y_pred_poly, y_pred_nn], ['L', 'P', 'N'])

**Read the plot:**
- The `·` dots are the true data (the parabola).
- The `L`s trace a straight line — linear regression can ONLY draw lines.
- The `P`s follow the parabola — polynomial regression nails it.
- The `N`s also follow the parabola — neural network also nails it.

Both `P` and `N` succeed. The `L` is doomed.

## Step 8 — The big reveal: what IS BERT, then?

Now we can answer the question that may have been confusing.

**BERT, GPT, PRAGMA — they are NEURAL NETWORKS.** Specifically, a kind called **Transformers**.

A Transformer is built out of:

```
input tokens
   │
   ▼
Embedding      ─►  linear lookup (one of the only LINEAR-only parts)
   │
   ▼
Attention      ─►  multiple LINEAR projections (Q, K, V) + softmax (NONLINEAR) + matmul
   │
   ▼
Feed-forward   ─►  Linear → GELU (NONLINEAR) → Linear     ← this is just an MLP!
   │
   ▼
(repeat)
   │
   ▼
Output head    ─►  Linear projection to vocabulary scores
```

The key part: there are **nonlinearities** sprinkled throughout (softmax inside attention, GELU inside the feed-forward sub-layer). Without them, the whole thing would collapse to one big linear function, just like stacking two Linear layers without a ReLU collapses to a single Linear.

**So no — BERT is not linear regression. BERT is a neural network with attention layers.**

What's the same:
- ✅ The 5-line training loop (predict → loss → backward → step)
- ✅ Gradient descent
- ✅ Embeddings as the input layer
- ✅ A Linear "head" at the output

What's different:
- ❌ The middle of the model: linear regression has `w*x + b`; BERT has stacked attention + feed-forward layers with nonlinearities.

Same training. Different model.

## Step 9 — The complete picture: model family tree

| Model | Architecture | Parameters | Where it shines |
|---|---|---|---|
| **Linear regression** (L1) | `y = w*x + b` | 2 | Predicting a single number from a single number, when the relationship is linear |
| **Polynomial regression** (L1.5 above) | `y = w₁x + w₂x² + ... + b` | 3 to N | Same, when relationship is polynomial and you know what features to add |
| **MLP / Neural net** (L1.5 above) | Linear → ReLU → Linear → (...) | tens to thousands | Predicting any function of any feature vector, when the features can be learned |
| **CNN** (not in this course) | Convolutions + ReLU + pooling | millions | Images |
| **RNN / LSTM** (briefly in L3b) | Recurrent steps over a sequence | millions | Sequences (but with memory issues — see L3b) |
| **Transformer** (L4, L5, BERT, PRAGMA, GPT) | Attention + feed-forward + LayerNorm + residual | millions to **trillions** | Sequences, where every token can look at every other token |

**Every single one is trained with the SAME 5-line loop.** Only the model class changes.

## Step 10 — Inspect what the neural net "learned"

Just for fun: peek inside the trained MLP and see what each hidden unit responds to.

In [ ]:
# Run the trained MLP and look at the hidden activations
with torch.no_grad():
    hidden = F.relu(net.layer1(x_in))   # (40, 8)

print("Hidden unit activations across the input range:")
print(f"  Each row = one input x; each column = one of the 8 hidden units")
print(f"  '·' means 0 (ReLU clipped it); '█' means active")
print()
print(f"  {'x':>6s} | " + " ".join(f"h{i}" for i in range(8)))
print("  " + "-" * 30)
for i in [0, 5, 10, 15, 20, 25, 30, 35, 39]:
    row = ""
    for h in hidden[i].tolist():
        row += " " + ("█" if h > 0.5 else ("▄" if h > 0.01 else "·"))
    print(f"  {x_data[i].item():>6.2f} |{row}")
print()
print("Each hidden unit has learned to respond to a different part of the x range.")
print("Some fire for negative x, some for positive x, some for the middle.")
print("Adding them up (with the weights learned in layer2) reconstructs the parabola.")

## Step 11 — Things to try

### 🟢 Easy

1. **Different curves.** Replace `y_true = x² - 4x + 3` with `y_true = torch.sin(x_data)` (or any other curve you want). Re-train all three models. Polynomial regression will fail unless you add more features; the neural network will succeed automatically.

2. **More hidden units.** Change `MLP(hidden=8)` to `MLP(hidden=64)`. Does it train better? Worse? Faster? Slower?

### 🟡 Medium

3. **Deeper network.** Add another hidden layer:
   ```python
   self.layer1 = nn.Linear(1, 8)
   self.layer2 = nn.Linear(8, 8)
   self.layer3 = nn.Linear(8, 1)
   def forward(self, x):
       h = F.relu(self.layer1(x))
       h = F.relu(self.layer2(h))
       return self.layer3(h)
   ```
   What does deeper buy you? (On this simple problem, not much. On real problems, depth is essential.)

4. **Replace ReLU with Sigmoid or Tanh.** Try `torch.sigmoid` or `torch.tanh` instead of `F.relu`. Compare training curves.

### 🔴 Hard

5. **What happens with NO activation function?** Remove the `F.relu` line. Now the MLP collapses to a single linear function. Train it and verify it does no better than linear regression — this is the "without nonlinearity, depth doesn't help" lesson.

6. **A "neural network" with attention.** Replace the MLP's hidden layer with a `nn.MultiheadAttention` layer. (You'll need to reshape your input to `(batch, seq=1, features)`.) This is the smallest possible Transformer-like model. Train it and verify it can also fit the parabola — though attention is wildly overkill for a 1D scalar regression.


## Summary

You now know:

- ✅ Linear regression fits **lines** (only).
- ✅ Polynomial regression (still "linear regression" mathematically) fits **polynomials** if you handcraft x² and x³ as features.
- ✅ A neural network with a nonlinearity (ReLU, GELU, etc.) can fit **any smooth function** without needing handcrafted features — it learns them.
- ✅ **BERT, PRAGMA, GPT are neural networks** (specifically Transformers). Their power comes from learnable embeddings + many attention layers + nonlinearities. They are NOT linear regression.
- ✅ The **5-line training loop is the same** for all of these. What changes is the model class.

Open [Lesson 4](lesson_04_tiny_bert.ipynb) again with fresh eyes — that tiny BERT is a neural network with about 3000 parameters that uses attention. Everything before that lesson was warm-up.
